# quant-retrieval on a free GPU

Runs training and evaluation on Colab's T4 instead of a laptop. Everything here
is a thin wrapper: the logic lives in `scripts/` and `src/` in the repo, and this
notebook only clones, restores two artifacts, and runs commands. Nothing is
implemented here, on purpose, so the cloud and the laptop run identical code.

## One-time setup, before the first run

Put these two things in a folder called `quant-retrieval` at the top level of
your Google Drive:

    quant-retrieval/
      minilm_tuned_epoch3/     <- the whole folder from checkpoints/minilm_tuned/epoch-3
      negatives.parquet        <- from data/processed/negatives.parquet

That is about 88 MB and you only do it once. Everything else, including the
26,152 document corpus, gets rebuilt here in about a minute, because the data
pipeline is deterministic and runs on CPU.

The tuned checkpoint is uploaded rather than retrained because it was trained on
the laptop's GPU. Retraining it here would produce slightly different weights and
every number already committed would stop being comparable.

## Each run

Runtime, Change runtime type, T4 GPU. Then Runtime, Run all. The last cell prints
the result files to copy back into the repo.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || print("NO GPU: set Runtime > Change runtime type > T4 GPU")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
ARTIFACTS = Path('/content/drive/MyDrive/quant-retrieval')
assert ARTIFACTS.exists(), f"create {ARTIFACTS} in your Drive first, see the notes above"
print(sorted(p.name for p in ARTIFACTS.iterdir()))

In [ ]:
%cd /content
!rm -rf quant-retrieval
!git clone -q https://github.com/melihgiray/quant-retrieval.git
%cd /content/quant-retrieval

# Colab already ships torch built for this GPU. Installing our pinned version
# would replace it with a slower or broken build, so install the package without
# its dependencies and add only what Colab lacks.
!pip install -q py7zr
!pip install -q -e . --no-deps

import importlib
missing = [m for m in ("pandas","numpy","scipy","torch","transformers","yaml","bs4","lxml","tqdm","pyarrow")
           if not importlib.util.find_spec(m)]
print("missing:", missing or "nothing")
import torch; print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

In [ ]:
# Rebuild the dataset. Deterministic and CPU only, so it matches the laptop byte
# for byte. About a minute.
!python scripts/download_data.py
!python scripts/build_dataset.py 2>&1 | tail -5

In [ ]:
# Restore the two things that cannot be rebuilt here.
!mkdir -p checkpoints/minilm_tuned
!cp -r "/content/drive/MyDrive/quant-retrieval/minilm_tuned_epoch3" checkpoints/minilm_tuned/epoch-3
!cp "/content/drive/MyDrive/quant-retrieval/negatives.parquet" data/processed/negatives.parquet
!ls checkpoints/minilm_tuned/epoch-3 && ls -la data/processed/negatives.parquet

## What to run

Edit this list. Each entry is a shell command run in order, and the run stops at
the first failure so a broken step does not look like a clean sweep.

In [ ]:
COMMANDS = [
    # Train the reranker on a mix of mined and random negatives.
    "python scripts/train_reranker.py --config configs/reranker_mixed.yaml",

    # Probe it against random documents BEFORE the expensive evaluations. If this
    # is near chance the pipeline numbers are already decided and the rest is
    # just confirmation.
    "python scripts/probe_reranker.py --checkpoint checkpoints/reranker_mixed/epoch-2",

    # The three pipelines.
    "python scripts/evaluate.py --config configs/bm25_rerank_mixed.yaml",
    "python scripts/evaluate.py --config configs/dense_rerank_mixed.yaml",
    "python scripts/evaluate.py --config configs/hybrid_rerank_mixed.yaml",
]

import subprocess, sys, time
for command in COMMANDS:
    print("=" * 70, "\n", command, flush=True)
    started = time.time()
    completed = subprocess.run(command, shell=True)
    print(f"[{time.time() - started:.0f}s, exit {completed.returncode}]", flush=True)
    if completed.returncode != 0:
        sys.exit(f"stopped: {command}")
print("\nALL DONE")

In [ ]:
# Copy everything worth keeping back to Drive.
!mkdir -p "/content/drive/MyDrive/quant-retrieval/out"
!cp -r results "/content/drive/MyDrive/quant-retrieval/out/"
!cp -r checkpoints "/content/drive/MyDrive/quant-retrieval/out/" 2>/dev/null || true
print("checkpoints and results saved to Drive under quant-retrieval/out")

In [ ]:
# Print the small files to copy back into the repo. Results JSONs are a few KB;
# checkpoints stay in Drive and never need to come down.
import json, pathlib
for path in sorted(pathlib.Path("results").glob("*.json")):
    if path.stat().st_mtime > pathlib.Path("README.md").stat().st_mtime:
        print("=" * 70, "\n", path)
        print(json.dumps(json.loads(path.read_text()), indent=2, sort_keys=True)[:2000])